# Spatial Phenotype Enrichment

This notebook extends the spatial phenotype interaction summary by adding abundance-aware normalization. Raw neighbor counts show how often two phenotypes are adjacent, but raw counts are strongly influenced by how common each phenotype is. A common phenotype can have many neighbor edges simply because many cells of that phenotype exist.

Spatial enrichment asks a more specific question:

```text
Is a phenotype pair observed more often than expected from phenotype abundance?
```

Main inputs:

```text
results/13_cells_with_phenotypes.csv
results/17_spatial_phenotype_interactions_by_image.csv
results/18_spatial_phenotype_interactions_by_category.csv
```

Main outputs:

```text
results/20_spatial_phenotype_enrichment_by_image.csv
results/21_spatial_phenotype_enrichment_by_category.csv
figures/07_spatial_phenotype_enrichment_heatmap.png
```

Important interpretation boundary:

- This step uses rule-based approximate phenotypes.
- Enrichment values describe spatial adjacency relative to abundance.
- Enrichment does not prove direct biological communication.
- Formal statistical testing or permutation analysis is required before making strong biological claims.


## Step 0: Configure Workflow Paths

This setup cell defines the input and output paths for the enrichment analysis.

Required inputs:

- The phenotyped cell table provides phenotype abundance by image and category.
- The image-level interaction table provides observed phenotype-neighbor edge counts per ROI.
- The category-level interaction table provides observed phenotype-neighbor edge counts per sample category.

The enrichment script combines these inputs to calculate expected edge fractions from phenotype abundance.


In [ ]:
from pathlib import Path
import csv
import subprocess

current = Path.cwd().resolve()
if current.name == 'notebooks':
    WORKFLOW_DIR = current.parent
else:
    WORKFLOW_DIR = current

RESULTS_DIR = WORKFLOW_DIR / 'results'
FIGURES_DIR = WORKFLOW_DIR / 'figures'
SCRIPTS_DIR = WORKFLOW_DIR / 'scripts'

PHENOTYPED_CELLS = RESULTS_DIR / '13_cells_with_phenotypes.csv'
INTERACTIONS_IMAGE = RESULTS_DIR / '17_spatial_phenotype_interactions_by_image.csv'
INTERACTIONS_CATEGORY = RESULTS_DIR / '18_spatial_phenotype_interactions_by_category.csv'

print('Workflow directory:', WORKFLOW_DIR)
print('Phenotyped cell table exists:', PHENOTYPED_CELLS.exists())
print('Image-level interaction table exists:', INTERACTIONS_IMAGE.exists())
print('Category-level interaction table exists:', INTERACTIONS_CATEGORY.exists())
print('Enrichment script exists:', (SCRIPTS_DIR / '12_spatial_phenotype_enrichment.py').exists())


## Step 1: Understand Observed and Expected Spatial Interactions

The previous notebook counted observed phenotype-neighbor edges. For example, a row may report the number of `Plasma_cell_like -- T_cell_like` neighbor edges in one ROI image.

Raw observed edge count:

```text
number of neighbor edges for a phenotype pair
```

Observed edge fraction:

```text
edge_count / all phenotype-neighbor edges in the image or category
```

Expected edge fraction from abundance:

The expected edge fraction is calculated from phenotype frequencies in the same image or category. If phenotype A represents 20% of cells and phenotype B represents 10% of cells, then a mixed A-B edge is expected at approximately:

```text
2 * 0.20 * 0.10 = 0.04
```

The factor of 2 is used for mixed phenotype pairs because an undirected edge can contain A-B or B-A ordering. For same-phenotype pairs, the expected fraction is:

```text
p(A) * p(A)
```

This expected value is a simple abundance-based baseline. It does not account for tissue architecture, cell size, segmentation shape, or local density differences.


## Step 2: Understand Enrichment Metrics

The script reports several metrics for each phenotype pair.

Columns in the enrichment outputs:

```text
group: Image or category
phenotype_a: first phenotype in the pair
phenotype_b: second phenotype in the pair
edge_count: observed unique undirected neighbor edges
observed_edge_fraction: observed fraction of all edges in the group
expected_edge_fraction_from_abundance: expected fraction based on phenotype abundance
observed_expected_ratio: observed fraction divided by expected fraction
log2_enrichment: log2(observed / expected)
group_cell_count: total cells in the image or category
phenotype_a_cell_count: number of cells with phenotype_a
phenotype_b_cell_count: number of cells with phenotype_b
```

How to interpret `observed_expected_ratio`:

```text
> 1: observed more often than expected from abundance
= 1: observed approximately as expected from abundance
< 1: observed less often than expected from abundance
```

How to interpret `log2_enrichment`:

```text
positive value: enriched adjacency
zero: approximately expected adjacency
negative value: depleted adjacency
```

Example:

```text
log2_enrichment = 1 means observed/expected = 2-fold enrichment
log2_enrichment = -1 means observed/expected = 2-fold depletion
```


## Step 3: Run Spatial Phenotype Enrichment

The script performs the following operations:

```text
1. Read the phenotyped single-cell table.
2. Count phenotype abundance within each image and category.
3. Read observed phenotype-neighbor interactions by image and category.
4. Calculate expected edge fractions from phenotype abundance.
5. Calculate observed/expected ratios.
6. Calculate log2 enrichment values.
7. Save image-level and category-level enrichment tables.
8. Save a heatmap of spatial phenotype enrichment.
```

The image-level table is useful for ROI-specific review. The category-level table is useful for compact exploratory comparison across `MGUS`, `UB`, and `B`.


In [ ]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '12_spatial_phenotype_enrichment.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--cells', 'results/13_cells_with_phenotypes.csv',
    '--interactions-image', 'results/17_spatial_phenotype_interactions_by_image.csv',
    '--interactions-category', 'results/18_spatial_phenotype_interactions_by_category.csv',
    '--output-image', 'results/20_spatial_phenotype_enrichment_by_image.csv',
    '--output-category', 'results/21_spatial_phenotype_enrichment_by_category.csv',
    '--heatmap', 'figures/07_spatial_phenotype_enrichment_heatmap.png',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)


## Step 4: Review Image-Level Enrichment

The image-level enrichment table contains one row per observed phenotype pair in each ROI image.

This table is useful for identifying ROI-specific spatial organization. For example, a phenotype pair may be enriched in one ROI but not in another. Such differences can reflect true tissue heterogeneity, differences in cell abundance, segmentation effects, or conservative phenotype rules.

Interpretation should focus on both the enrichment value and the supporting edge count. High enrichment based on very few edges is less stable than enrichment supported by many edges.


In [ ]:
image_enrichment = RESULTS_DIR / '20_spatial_phenotype_enrichment_by_image.csv'
with image_enrichment.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)

for row in rows[:30]:
    print(row)


## Step 5: Review Category-Level Enrichment

The category-level enrichment table aggregates observed interactions and abundance baselines by category.

This table provides a compact exploratory overview of phenotype-neighbor enrichment across `MGUS`, `UB`, and `B`. It should be interpreted cautiously because the representative subset is small and phenotype labels are rule-based approximations.

Useful review approach:

- Sort by high positive `log2_enrichment` to find enriched phenotype pairs.
- Sort by negative `log2_enrichment` to find depleted phenotype pairs.
- Check `edge_count` to avoid over-interpreting rare pairs.
- Compare enrichment with raw composition and interaction tables.


In [ ]:
category_enrichment = RESULTS_DIR / '21_spatial_phenotype_enrichment_by_category.csv'
with category_enrichment.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)

for row in rows[:40]:
    print(row)


## Step 6: Review Enrichment Heatmap

The heatmap summarizes log2 enrichment values across phenotype pairs. Red-blue style coloring is used so enriched and depleted relationships are visually separated.

Interpretation guidance:

- Positive values indicate more neighbor edges than expected from phenotype abundance.
- Negative values indicate fewer neighbor edges than expected from phenotype abundance.
- Values near zero indicate approximately abundance-expected adjacency.
- Diagonal values describe same-phenotype adjacency.
- Off-diagonal values describe mixed-phenotype adjacency.

Important limitation:

This heatmap is an abundance-normalized exploratory summary, not a formal spatial statistics test. Strong biological interpretation requires validation with robust phenotype annotation, larger sample context, and preferably permutation-based spatial enrichment testing.


In [ ]:
heatmap = FIGURES_DIR / '07_spatial_phenotype_enrichment_heatmap.png'
print('Heatmap exists:', heatmap.exists())
print('Heatmap size_bytes:', heatmap.stat().st_size if heatmap.exists() else 0)
